In [1]:
import pandas as pd
from gotrackit.gps.Trajectory import TrajectoryPoints
import time

import numpy as np

In [9]:
import os

save_path = r"D:\Research\GoTrackit\Apr17_hired"
folder = os.path.dirname(save_path)

os.makedirs(folder, exist_ok=True)

In [ ]:
a = np.array((1,2,3))


In [3]:
import geopandas as gpd
from gotrackit.map.Net import Net

# 读取路网数据
link = gpd.read_file(r'./data/input/Shanghai_gotrackit_links_updated.shp')
node = gpd.read_file(r'./data/input/Shanghai_gotrackit_nodes_updated.shp')
my_net = Net(link_gdf=link, node_gdf=node, not_conn_cost=1200)
my_net.init_net()  # net初始化

__init__ costs :0.3178236484527588 seconds!


In [5]:
from gotrackit.MapMatch import MapMatch
c = time.time()
gps_df = pd.read_csv(r'./data/input/taxi_data_gotracki_apr17_hired.csv')
gps_df

,agent_id,lng,lat,time
0,2,121.428460,31.273712,2015-04-14 05:30:04
1,2,121.428460,31.273712,2015-04-14 05:30:14
2,2,121.428460,31.273712,2015-04-14 05:30:24
3,2,121.428460,31.273712,2015-04-14 05:30:34
4,2,121.428460,31.273712,2015-04-14 05:30:44
...,...,...,...,...
14400057,30050,121.418667,31.211847,2015-04-14 11:29:17
14400058,30050,121.418667,31.211848,2015-04-14 11:29:27
14400059,30050,121.418663,31.211848,2015-04-14 11:29:37
14400060,30050,121.418663,31.211848,2015-04-14 11:29:47


In [ ]:
from gotrackit.MapMatch import MapMatch
c = time.time()
gps_df = pd.read_csv(r'./data/input/taxi_data_gotracki_apr17_empty.csv')

# 利用gps数据构建TrajectoryPoints, 并且对数据进行清洗
# 是否需要进行该步操作视实际情况而定
tp = TrajectoryPoints(gps_points_df=gps_df, plain_crs='EPSG:32649')
tp.lower_frequency(lower_n=2).kf_smooth(o_deviation=0.3)  # 由于样例数据定位频率高且有一定的误差，因此先做间隔采样然后执行滤波平滑
gps_df = tp.trajectory_data(_type='df')

a = time.time()

print(a-c)

# 构建匹配类
# 指定要输出HTML可视化文件
# 指定项目的标志字符flag_name='general_sample', 这个用户可以自定义
# gps数据时间列的值都是形如2022-05-12 16:27:46，因此指定时间列格式为 '%Y-%m-%d %H:%M:%S'
# 关于gps_buffer的确定，需要将路网和gps数据一同使用gis软件可视化，大概确定GPS数据和候选路段的距离
mpm = MapMatch(net=my_net, flag_name='general_sample', 
               time_format='%Y-%m-%d %H:%M:%S',
               gps_buffer=120, 
               use_heading_inf=True, omitted_l=6.0, 
               del_dwell=False, dense_gps=False,
               export_html=True, 
               out_fldr=r'D:/Research/GoTrackit/Apr17_empty')

# 执行匹配
# 第一个返回结果是匹配结果表
# 第二个是发生警告的agent的相关信息({agent_id1: pd.DataFrame(), agent_id2: pd.DataFrame()...})
# 第三个是匹配出错的agent的id列表(GPS点经过预处理(或者原始数据)后点数量不足2个)
match_res, warn_info, error_info = mpm.execute(gps_df=gps_df)
match_res.to_csv(fr'D:/Research/GoTrackit/Apr17_empty/apr17_empty_matched.csv',
                 encoding='utf_8_sig', index=False)

b = time.time()
print(b-a)

2078.9033076763153
- gotrackit ------> No.1: agent: 2 
using sub net
__init__ costs :0.0 seconds!
create_computational_net costs :0.12655019760131836 seconds!
do not use prj_cache


C:\Users\koich\AppData\Roaming\Python\Python312\site-packages\gotrackit\model\Markov.py:311: UserWarning: the GPS point with seq: [116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289,

__generate_st costs :0.37865519523620605 seconds!


C:\Users\koich\AppData\Roaming\Python\Python312\site-packages\gotrackit\model\Markov.py:1100: UserWarning: gps seq: 114 -> 115 problem with state transfer
                            from_link:(3, 75) -> to_link:(74, 35)
  warnings.warn(
C:\Users\koich\AppData\Roaming\Python\Python312\site-packages\gotrackit\model\Markov.py:311: UserWarning: the GPS point with seq: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 117, 118] is not associated with any candidate road segment 
                            and will not be used for path matching calculation...
  warnings.warn(rf'''the G

- gotrackit ------> No.2: agent: 4 
after data preprocessing, there are less than 2 GPS observation points.
- gotrackit ------> No.3: agent: 7 
using sub net
__init__ costs :0.0031976699829101562 seconds!
create_computational_net costs :0.007852792739868164 seconds!
do not use prj_cache
__generate_st costs :0.013036727905273438 seconds!
- gotrackit ------> No.4: agent: 11 
using sub net
__init__ costs :0.0 seconds!
create_computational_net costs :0.015620708465576172 seconds!
do not use prj_cache
__generate_st costs :0.12219429016113281 seconds!
- gotrackit ------> No.5: agent: 12 
using sub net
__init__ costs :0.003000020980834961 seconds!
create_computational_net costs :0.00800013542175293 seconds!
do not use prj_cache


...

C:\Users\koich\AppData\Roaming\Python\Python312\site-packages\gotrackit\model\Markov.py:1100: UserWarning: gps seq: 665 -> 666 problem with state transfer
                            from_link:(4375, 4380) -> to_link:(4554, 4408)
  warnings.warn(


__init__ costs :0.0 seconds!
create_computational_net costs :0.199843168258667 seconds!
do not use prj_cache


C:\Users\koich\AppData\Roaming\Python\Python312\site-packages\gotrackit\model\Markov.py:311: UserWarning: the GPS point with seq: [0, 1, 2, 3, 4, 5] is not associated with any candidate road segment 
                            and will not be used for path matching calculation...
  warnings.warn(rf'''the GPS point with seq: {_gap} is not associated with any candidate road segment


__generate_st costs :0.43813514709472656 seconds!
- gotrackit ------> No.1416: agent: 11683 
using sub net
__init__ costs :0.0 seconds!
create_computational_net costs :0.5002079010009766 seconds!
do not use prj_cache


C:\Users\koich\AppData\Roaming\Python\Python312\site-packages\gotrackit\model\Markov.py:311: UserWarning: the GPS point with seq: [316, 317, 49, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 320, 321, 319, 318] is not associated with any candidate road segment 
                            and will not be used for path matching calculation...
  warnings.warn(rf'''the GPS point with seq: {_gap} is not associated with any candidate road segment
